# Criando e Usando Pacotes em Python
### Apresentação prática usando o projeto **`pacotesoma`** como modelo

---

Este notebook é uma **apresentação executável**. Ele usa o projeto **`pacotesoma`** como **modelo real** ao longo de toda a explicação.

> **Estrutura Geral**
> ```text
> apresentacao_pacotesoma/
> ├── apresentacao_pacotes.ipynb    ← este notebook
> └── PacoteSoma/                   ← projeto-modelo que vamos usar como exemplo
> ```

**O que você vai aprender:**

1. O que é um pacote e por que ele existe
2. Diferença entre **módulo** e **pacote**
3. Como o projeto **`pacotesoma`** está organizado (e por quê)
4. O papel do arquivo `__init__.py`
5. Como construir um instalador `.whl` e um source dist `.tar.gz`
6. Como instalar, testar e publicar seu pacote
7. **Por que** cada técnica é usada — não só o "como"

> **Importante:** o notebook lê os arquivos do projeto `pacotesoma` em tempo real.
> Tudo que você vê nas células é o conteúdo verdadeiro daquela pasta.


## Setup inicial

```
Arquivos e pastas do projeto:
  📄 .gitignore
  📄 LICENSE
  📄 README.md
  📂 dist
  📂 pacotesoma
  📄 pyproject.toml
  📄 setup.cfg
  📄 setup.py
  📂 tests
```

## 1. Antes de tudo: uma analogia

Imagine que você está cozinhando.

- **Uma receita** = um arquivo `.py` (chamamos isso de **módulo**)
- **Um livro de receitas** = uma pasta com vários arquivos `.py` relacionados (isso é um **pacote**)
- **Uma biblioteca cheia de livros** = uma **biblioteca** Python (como `numpy`, `pandas`)

Quando seu projeto cresce e você passa de "uma receita" para "muitas receitas que se relacionam",
você organiza tudo num livro. Esse "livro" é o **pacote**.

### Definições simples

| Conceito | O que é fisicamente | No projeto `pacotesoma` |
|---|---|---|
| **Função** | Bloco de código que faz uma tarefa | `soma(a, b)` |
| **Módulo** | Um arquivo `.py` com funções | `pacotesoma/pacotesoma.py` |
| **Pacote** | Uma **pasta** com módulos | `pacotesoma/` |
| **Biblioteca** | Pacote distribuído publicamente | `numpy`, `pandas`, `RasterIO`... |


## 2. Por que criar um pacote?

"Se meu código já funciona em um arquivo `.py`, por que me preocupar com tudo isso?"

### 1. Organização
Conforme o projeto cresce, um arquivo gigante vira caos. Pacotes permitem separar responsabilidades.

### 2. Reaproveitamento
Um pacote bem feito pode ser instalado (`pip install meu_pacote`) e usado em **qualquer outro projeto**,
sem copiar e colar código.

### 3. Compartilhamento
Você pode publicar no [PyPI](https://pypi.org) e qualquer pessoa instala com `pip install`.
É assim que `numpy`, `pandas` e `requests` chegam até nós.

> **Para iniciantes:** mesmo que você nunca pretenda publicar nada, estruturar o código como pacote
> desde cedo evita problemas de importação, conflitos de nomes e dificuldades quando o projeto crescer.


## 3. A estrutura do `pacotesoma`

Vamos olhar exatamente como o projeto-modelo está organizado:

```
pacotesoma/
├── pacotesoma/
│   ├── __init__.py
│   └── pacotesoma.py
├── tests/
│   └── test_soma.py
├── LICENSE
├── pyproject.toml
├── README.md
├── setup.cfg
└── setup.py
```

### "Por que tem `pacotesoma/` dentro de `pacotesoma/`?"

Essa é a maior confusão de quem começa. A resposta:

- A pasta **externa** é o **projeto** (código + documentação + configuração).
- A pasta **interna** é o **pacote** em si (apenas o código Python que será instalado).

Quando você faz `pip install pacotesoma`, apenas a pasta **interna** vai parar no seu Python.
O resto (`README.md`, `tests/`, `setup.py`) fica só no projeto, no GitHub, etc.


## 4. O módulo: onde mora a função

Vamos olhar o conteúdo real de **`pacotesoma/pacotesoma.py`**:

In [ ]:
from pathlib import Path

PROJETO = Path("PacoteSoma").resolve() # Tomando path absoluto

arquivo_modulo = PROJETO / "pacotesoma" / "pacotesoma.py"
print(arquivo_modulo.read_text(encoding="utf-8"))

### O que aprender daqui

| Trecho | Por que está ali |
|---|---|
| `def soma(a, b):` | A função pública do pacote |
| `isinstance(a, (int, float))` | **Valida entradas**: evita comportamento inesperado se alguém passar uma string ou lista |
| Mensagem em string em vez de `raise` | Escolha de design — mantém a função simples para a demonstração |

> **Por que validar tipos?** Em Python, o operador `+` funciona em listas, strings, números...
> Sem validação, `soma("ola", "mundo")` retornaria `"olamundo"` silenciosamente. Validar deixa o
> comportamento da função previsível.


In [ ]:
dir()

In [ ]:
import PacoteSoma
print(type(PacoteSoma))

In [ ]:
dir()

In [ ]:
somar = PacoteSoma.pacotesoma.soma

In [ ]:
import PacoteSoma.pacotesoma as pct
print(type(pct))

In [ ]:
dir(PacoteSoma.pacotesoma)

In [ ]:
somar = PacoteSoma.pacotesoma.soma
somar(1,2)

In [ ]:
# from PacoteSoma.pacotesoma import soma
# soma(5,8)
# from PacoteSoma.pacotesoma import soma as somando
# somando(5,7)
# import PacoteSoma.pacotesoma as pct
# pct.soma(5,8)

In [ ]:
#Acesso ao módulo
from PacoteSoma.pacotesoma.pacotesoma import hello

hello()

In [ ]:
print(type(PacoteSoma.pacotesoma.pacotesoma))

In [ ]:
dir(PacoteSoma.pacotesoma.pacotesoma)

## 5. O `__init__.py`

Este é o arquivo que mais confunde iniciantes. Vamos descomplicar olhando o do projeto-modelo:

In [ ]:
arquivo_init = PROJETO / "pacotesoma" / "__init__.py"
print(arquivo_init.read_text(encoding="utf-8"))

### Existem os pacotes namespace que não necessitam de `__init__`

### O que cada linha significa

| Linha | Significado |
|---|---|
| `from .pacotesoma import soma` | Sobe a função `soma` do módulo interno para o nível do pacote |
| `__all__ = ["soma"]` | Lista do que é exportado em `from pacotesoma import *` |
| `__version__ = "0.1.0"` | Versão acessível como `pacotesoma.__version__` |

### Comparando o que o `__init__.py` faz pelo usuário

Sem o atalho dentro do `__init__.py`:
```python
from PacoteSoma.pacotesoma.pacotesoma import soma     # caminho longo
```

Com o atalho:
```python
from PacoteSoma.pacotesoma import soma                # limpo, direto
```

> **Por que isso importa?** Você pode renomear `pacotesoma.py` para qualquer coisa amanhã,
> e quem usa o pacote **não precisa mudar uma linha**. O `__init__.py` é a "fachada" que esconde
> a estrutura interna.


In [ ]:
from PacoteSoma.pacotesoma import soma 
soma (4,5)

## 6. Metadados: a identidade do pacote

Um pacote tem identidade: nome, versão, autor, licença... Existem **três formas** de declarar isso.
Vamos olhar as três, no projeto-modelo.

### 6.1 `pyproject.toml` — a forma moderna

É **declarativo** (sem código executável), padronizado, mais seguro.

In [ ]:
print((PROJETO / "pyproject.toml").read_text(encoding="utf-8"))

**Pontos a destacar:**

- `[build-system]` → qual ferramenta constrói o pacote (`setuptools` neste caso)
- `[project]` → identidade do pacote (nome, versão, autor, licença, dependências)
- `[project.optional-dependencies]` → grupos opcionais (ex: `pip install pacotesoma[dev]`)
- `[tool.setuptools.packages.find]` → diz para o setuptools **não** incluir a pasta `tests/`

### 6.2 `setup.py` — a forma antiga

É um **script Python**. Funciona, mas por ser executável, é menos seguro e mais difícil de analisar
por ferramentas automatizadas.

In [ ]:
print((PROJETO / "setup.py").read_text(encoding="utf-8"))

### 6.3 `setup.cfg` — alternativa declarativa

Mesma ideia do `pyproject.toml`, mas com sintaxe mais antiga. Você verá em projetos legados.

In [ ]:
print((PROJETO / "setup.cfg").read_text(encoding="utf-8"))

> **Por que existem três?** Histórico: `setup.py` veio primeiro, depois `setup.cfg` para deixar
> declarativo, e em 2021 o `pyproject.toml` virou padrão oficial (PEP 621).
> **Se você começa hoje, use `pyproject.toml`.** Os outros estão aqui no projeto apenas para você
> aprender a reconhecê-los em projetos existentes.


## 7. Documentação e licença

Faltam dois arquivos importantes.

### `README.md` — o cartão de visita

É a primeira coisa que alguém vê no GitHub ou PyPI. Veja o do `pacotesoma`:

In [ ]:
# Mostramos só as primeiras 40 linhas para não tomar a tela inteira
linhas = (PROJETO / "README.md").read_text(encoding="utf-8").splitlines()
print("\n".join(linhas[:40]))
print(f"\n... ({len(linhas)} linhas no total)")

### `LICENSE` — quem pode usar e como

Sem licença, juridicamente **ninguém** pode usar seu código. A **MIT** (usada aqui) é uma escolha
popular, simples e permissiva.

In [ ]:
print((PROJETO / "LICENSE").read_text(encoding="utf-8")[:400], "...")

> **Em dúvida sobre qual licença usar?** Visite [choosealicense.com](https://choosealicense.com).


## 8. Construindo o instalador (`.whl` e `.tar.gz`)

Hora de transformar o código solto em um pacote distribuível.

### O que são esses dois arquivos?

| Arquivo | Para que serve | Analogia |
|---|---|---|
| `.whl` (wheel) | Instalador **pronto** | Móvel já montado |
| `.tar.gz` (sdist) | **Código-fonte** compactado | Móvel em peças com manual |

>  **Por que ter os dois?** O `.whl` é rápido (padrão moderno). O `.tar.gz` permite que alguém leia,
> modifique ou instale em sistemas exóticos. Publicar ambos é boa prática.

### Construindo o `pacotesoma`

A forma moderna usa `python -m build`:

### OBS:. Limpar outputs!

In [1]:
from pathlib import Path
PROJETO = Path("PacoteSoma").resolve() # Tomando path absoluto
arquivo_modulo = PROJETO / "pacotesoma" / "pacotesoma.py"

In [4]:
import subprocess, sys

# Garante que a ferramenta 'build' está instalada
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "--break-system-packages", "build"],
    check=True,
)

# Roda o build dentro da pasta do pacotesoma
resultado = subprocess.run(
    [sys.executable, "-m", "build"],
    cwd=PROJETO,
    capture_output=True, text=True,
)

print("--- Últimas linhas do build ---")
print("\n".join(resultado.stdout.strip().splitlines()[-3:]))
print()
print("Arquivos gerados em pacotesoma/dist/:")
for f in sorted((PROJETO / "dist").iterdir()):
    print(f" {f.name}  ({f.stat().st_size:,} bytes)")

--- Últimas linhas do build ---
adding 'pacotesoma-0.1.0.dist-info/RECORD'
removing build/bdist.linux-x86_64/wheel
Successfully built pacotesoma-0.1.0.tar.gz and pacotesoma-0.1.0-py3-none-any.whl

Arquivos gerados em pacotesoma/dist/:
 pacotesoma-0.1.0-py3-none-any.whl  (4,504 bytes)
 pacotesoma-0.1.0.tar.gz  (4,224 bytes)


In [5]:
dir()

['In',
 'Out',
 'PROJETO',
 'Path',
 '_',
 '_3',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '__vsc_ipynb_file__',
 '_dh',
 '_i',
 '_i1',
 '_i2',
 '_i3',
 '_i4',
 '_i5',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'arquivo_modulo',
 'exit',
 'f',
 'get_ipython',
 'open',
 'quit',
 'resultado',
 'subprocess',
 'sys']

In [13]:
import PacoteSoma.pacotesoma as ps 

In [14]:
ps.soma(4,5)

9

In [15]:
dir()

['In',
 'Out',
 'PROJETO',
 'PacoteSoma',
 'Path',
 '_',
 '_14',
 '_3',
 '_5',
 '_7',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '__vsc_ipynb_file__',
 '_dh',
 '_i',
 '_i1',
 '_i10',
 '_i11',
 '_i12',
 '_i13',
 '_i14',
 '_i15',
 '_i2',
 '_i3',
 '_i4',
 '_i5',
 '_i6',
 '_i7',
 '_i8',
 '_i9',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'arquivo_modulo',
 'exit',
 'f',
 'get_ipython',
 'open',
 'ps',
 'quit',
 'resultado',
 'subprocess',
 'sys']

## 9. Instalando localmente para testar

Antes de publicar para o mundo, sempre teste a instalação na sua máquina.

| Comando | Quando usar |
|---|---|
| `pip install .` | Instalação normal (vai para `site-packages`) |
| `pip install -e .` | **Modo desenvolvimento**: mudanças no código têm efeito imediato |
| `pip install dist/pacotesoma-0.1.0-py3-none-any.whl` | Testa o arquivo que será distribuído |

### Por que o modo `-e` (editável) existe?

Sem `-e`, você teria que reinstalar a cada edição. Com `-e`, o Python aponta direto para a sua
pasta de desenvolvimento — edita e roda imediatamente.

Vamos instalar a partir do `.whl` recém-gerado:

In [ ]:
import subprocess, sys

whl_path = next((PROJETO / "dist").glob("*.whl"))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--break-system-packages",
     "--force-reinstall", "--quiet", str(whl_path)],
    check=True,
)
print(f"Instalado a partir de: {whl_path.name}")

## 10. Usando o pacote

Agora podemos importar e usar como qualquer biblioteca:

In [19]:
# Limpa o cache de módulos para garantir que pegamos a versão recém-instalada
import sys
for nome in list(sys.modules):
    if nome.startswith("pacotesoma"):
        del sys.modules[nome]

from PacoteSoma.pacotesoma import soma
import PacoteSoma.pacotesoma

print("Versão do pacote:", PacoteSoma.pacotesoma.__version__)
print()
print("soma(4, 9)     =", soma(4, 9))
print("soma(1.5, 2.5) =", soma(1.5, 2.5))
print("soma(-3, 8)    =", soma(-3, 8))
print('soma("a", 2)   =', soma("a", 2))

Versão do pacote: 0.1.0

soma(4, 9)     = 13
soma(1.5, 2.5) = 4.0
soma(-3, 8)    = 5
soma("a", 2)   = Only integer or float numbers allowed


### A documentação aparece automaticamente

Lembra do **docstring**? O Python o usa como documentação:

In [20]:
help(soma)

Help on function soma in module PacoteSoma.pacotesoma.pacotesoma:

soma(a, b)
    Soma dois números (int ou float) e retorna o resultado.

    Parâmetros
    ----------
    a : int | float
        Primeiro número a ser somado.
    b : int | float
        Segundo número a ser somado.

    Retorno
    -------
    int | float
        A soma de ``a`` e ``b`` quando ambos são numéricos.
    str
        Mensagem de erro caso algum dos argumentos não seja numérico.

    Exemplos
    --------
    >>> soma(4, 9)
    13
    >>> soma(1.5, 2.5)
    4.0
    >>> soma("a", 2)
    'Only integer or float numbers allowed'



## 11. Testes automatizados — por que se importar?

Imagine que daqui a 3 meses você decide melhorar a função `soma`. Como ter certeza de que não
quebrou nada? **Testes automatizados.**

Você escreve exemplos verificáveis do que a função deve fazer. Se o comportamento mudar
sem querer, o teste falha — você descobre **antes** dos usuários.

### O arquivo de testes do `pacotesoma`

In [ ]:
print((PROJETO / "tests" / "test_soma.py").read_text(encoding="utf-8"))

### Rodando os testes com `pytest`

O `pytest` procura arquivos `test_*.py` e funções `test_*()` automaticamente.

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "--break-system-packages", "pytest"],
    check=True,
)

resultado = subprocess.run(
    [sys.executable, "-m", "pytest", "-v"],
    cwd=PROJETO,
    capture_output=True, text=True,
)
print(resultado.stdout)

> **Por que isso é especialmente importante para iniciantes?**
> Testes te obrigam a **pensar nos casos extremos**: e se a entrada for negativa? Zero? Uma string?
> Esse exercício te faz escrever código mais robusto.


## 12. Publicando no PyPI

Quando o pacote estiver pronto, você pode disponibilizá-lo para o mundo no
[Python Package Index (PyPI)](https://pypi.org/).

### Passo a passo

1. **Crie conta** em [pypi.org](https://pypi.org) e em [test.pypi.org](https://test.pypi.org).
2. **Instale o `twine`**:
   ```bash
   pip install twine
   ```
3. **Verifique** se os arquivos estão bem formados:
   ```bash
   python -m twine check dist/*
   ```
4. **Publique primeiro no TestPyPI** (ambiente de teste):
   ```bash
   python -m twine upload --repository testpypi dist/*
   ```
5. **Quando estiver tudo certo, publique no PyPI real:**
   ```bash
   python -m twine upload dist/*
   ```

### Por que usar TestPyPI antes?

Porque **no PyPI real você não pode reutilizar o mesmo número de versão**.
Se publicar `0.1.0` com bug, terá que lançar `0.1.1`. O TestPyPI permite ensaiar o processo
sem "queimar" versões.

> ⚠️ **Este notebook não publica de verdade.** Os comandos acima são para você executar no terminal
> quando quiser publicar seu próprio pacote.

## 13. Resumo visual do fluxo completo

```
1. Criar estrutura de pastas
        ↓
2. Escrever o módulo (.py com funções)
        ↓
3. Criar __init__.py (expõe a API pública)
        ↓
4. Configurar metadados (pyproject.toml)
        ↓
5. Adicionar README.md e LICENSE
        ↓
6. Escrever testes (tests/)
        ↓
7. Construir: python -m build
        ↓
8. Testar localmente: pip install dist/*.whl
        ↓
9. Validar: python -m twine check dist/*
        ↓
10. Publicar: python -m twine upload dist/*
```


## 14. Boas práticas ✅

| Prática | Por quê | No `pacotesoma`? |
|---|---|---|
| Nomes curtos e descritivos | Facilita lembrar e digitar | ✅ `pacotesoma` |
| Comece com `pyproject.toml` | Padrão moderno e seguro | ✅ Presente |
| Sempre escreva um `README.md` | Cartão de visita do projeto | ✅ Com exemplos |
| Sempre escreva testes | Mesmo simples valem muito | ✅ 8 testes |
| Versionamento semântico (`MAJOR.MINOR.PATCH`) | Convenção universal | ✅ `0.1.0` |
| `pip install -e .` no desenvolvimento | Evita reinstalar a cada edição | — |
| TestPyPI antes do PyPI real | Versão não é reaproveitável | — |
| Docstrings em todas as funções | Documentação automática via `help()` | ✅ Em `soma` |
| Adicione um `LICENSE` | Sem licença, ninguém pode usar legalmente | ✅ MIT |
| `__init__.py` simples | Só importações e metadados | ✅ 4 linhas |


## 15. Limpeza (opcional)

Se quiser, desinstale o pacote de teste:

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y",
     "--break-system-packages", "pacotesoma"],
    capture_output=True,
)
print("Pacote desinstalado. A pasta do projeto continua intacta para você estudar.")

## 16. Recapitulação do uso de cada técnica

| Técnica vista no `pacotesoma` | Motivo |
|---|---|
| Pasta dentro de pasta (`pacotesoma/pacotesoma/`) | Separar projeto (config + docs) do pacote (só código) |
| `__init__.py` com `from .pacotesoma import soma` | Esconde a estrutura interna; permite refatorar sem quebrar usuários |
| `pyproject.toml` em vez de só `setup.py` | Declarativo, padronizado, mais seguro |
| Construir `.whl` **e** `.tar.gz` | Cobre instalação rápida e instalação a partir do código-fonte |
| Testes em `tests/` separados do código | Não vão para o pacote final; ficam apenas no repositório |
| `__version__` no `__init__.py` | Permite saber em runtime qual versão está instalada |
| `LICENSE` (MIT) | Define direitos legais — sem ela, ninguém pode usar |
| `pip install -e .` no desenvolvimento | Edições refletem imediatamente sem reinstalar |
| `twine check` antes do upload | Detecta problemas no metadado antes de publicar |

---
